# E-Commerce Sales, Customer & Profitability Analytics

**Portfolio project:** E-Commerce Sales, Customer & Profitability Analytics  
**Tools:** Python, Pandas, NumPy, Matplotlib, SQL, Power BI  
**Purpose:** Analyze sales performance, product profitability, customer behavior, marketing performance, returns, and operations.

This notebook is designed for **Google Colab**. Run it from top to bottom. The analysis is calculated directly from the uploaded dataset, so KPIs are not hard-coded.

## 1. Upload the dataset

Upload the main file:

`ecommerce_sales_customer_analytics_150k...csv`

You can also upload the supporting files:
- `product_catalog...csv`
- `order_items...csv`
- `customer_master...csv`
- `dataset_statistics...csv`

The main dataset is enough to run the core analysis. Supporting files unlock product-level/category analysis.

In [ ]:
from google.colab import files
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

uploaded = files.upload()
uploaded_names = list(uploaded.keys())

print("Uploaded files:")
for name in uploaded_names:
    print(" -", name)

main_file = next(
    (f for f in uploaded_names if f.lower().startswith("ecommerce_sales_customer_analytics_150k") and f.lower().endswith(".csv")),
    None
)

if main_file is None:
    raise FileNotFoundError(
        "Main dataset not found. Please upload the ecommerce_sales_customer_analytics_150k CSV."
    )

df = pd.read_csv(main_file)

print(f"\nLoaded: {main_file}")
print("Rows:", len(df))
print("Columns:", len(df.columns))

## 2. Understand the dataset

First, inspect the structure, data types, sample records, and duplicate rows. This gives us a clean baseline before doing any business analysis.

In [ ]:
print("Shape:", df.shape)
display(df.head())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nExact duplicate rows:", df.duplicated().sum())
print("Unique order IDs:", df["order_id"].nunique())
print("Unique customers:", df["customer_id"].nunique())

## 3. Missing-value analysis

Missing values are not automatically errors. For example, a missing coupon code does not necessarily mean that no coupon was used, so we will report missingness rather than blindly filling values.

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100)
}).sort_values("missing_count", ascending=False)

display(missing[missing["missing_count"] > 0])

## 4. Prepare dates and analysis fields

We convert the order date to a proper datetime and create reusable time fields. We also create flags for returned and loss-making orders without changing the original business values.

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

numeric_cols = [
    "quantity", "gross_sales", "discount_amount", "tax_amount",
    "shipping_cost", "net_sales", "product_cost", "profit",
    "profit_margin_percentage", "customer_lifetime_value",
    "loyalty_points_earned", "loyalty_points_redeemed",
    "delivery_days", "estimated_delivery_days", "customer_rating"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["order_month"] = df["order_date"].dt.to_period("M").astype(str)
df["order_year"] = df["order_date"].dt.year
df["order_month_name"] = df["order_date"].dt.month_name()
df["is_returned"] = df["return_status"].eq("Returned")
df["is_cancelled"] = df["order_status"].eq("Cancelled")
df["is_loss_making"] = df["profit"] < 0

print("Date range:", df["order_date"].min().date(), "to", df["order_date"].max().date())

## 5. Executive KPI dashboard

These are the core business KPIs we will later reproduce in Power BI.

In [ ]:
total_orders = df["order_id"].nunique()
total_customers = df["customer_id"].nunique()
total_revenue = df["net_sales"].sum()
total_profit = df["profit"].sum()
profit_margin = (total_profit / total_revenue * 100) if total_revenue else np.nan
aov = total_revenue / total_orders if total_orders else np.nan
return_rate = df["is_returned"].sum() / total_orders * 100 if total_orders else np.nan
cancellation_rate = df["is_cancelled"].sum() / total_orders * 100 if total_orders else np.nan
loss_orders = df["is_loss_making"].sum()
loss_order_pct = loss_orders / total_orders * 100 if total_orders else np.nan

kpis = pd.DataFrame({
    "KPI": [
        "Total Orders", "Total Customers", "Net Sales",
        "Total Profit", "Profit Margin", "Average Order Value",
        "Return Rate", "Cancellation Rate",
        "Loss-Making Orders", "Loss-Making Order %"
    ],
    "Value": [
        total_orders, total_customers, total_revenue,
        total_profit, profit_margin, aov,
        return_rate, cancellation_rate,
        loss_orders, loss_order_pct
    ]
})

display(kpis)

## 6. Data-quality checks

These checks are intended to identify issues worth investigating. Negative profit is treated as a business outcome rather than automatically being deleted.

In [ ]:
quality_checks = pd.DataFrame({
    "Check": [
        "Missing order IDs",
        "Missing customer IDs",
        "Missing order dates",
        "Negative quantity",
        "Negative net sales",
        "Negative product cost",
        "Negative delivery days",
        "Negative estimated delivery days",
        "Negative profit / loss-making orders"
    ],
    "Count": [
        df["order_id"].isna().sum(),
        df["customer_id"].isna().sum(),
        df["order_date"].isna().sum(),
        (df["quantity"] < 0).sum(),
        (df["net_sales"] < 0).sum(),
        (df["product_cost"] < 0).sum(),
        (df["delivery_days"] < 0).sum(),
        (df["estimated_delivery_days"] < 0).sum(),
        (df["profit"] < 0).sum()
    ]
})

display(quality_checks)

## 7. Monthly sales and profit trend

A monthly trend helps identify seasonality, growth/decline periods, and unusually strong or weak months.

In [ ]:
monthly = (
    df.groupby("order_month", as_index=False)
      .agg(
          revenue=("net_sales", "sum"),
          profit=("profit", "sum"),
          orders=("order_id", "nunique")
      )
      .sort_values("order_month")
)

display(monthly.head(12))

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(monthly["order_month"], monthly["revenue"])
plt.title("Monthly Net Sales")
plt.xlabel("Month")
plt.ylabel("Net Sales")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(monthly["order_month"], monthly["profit"])
plt.title("Monthly Profit")
plt.xlabel("Month")
plt.ylabel("Profit")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## 8. Sales channel and customer segment analysis

This section compares revenue and profitability across the main commercial segments.

In [ ]:
channel_performance = (
    df.groupby("sales_channel", as_index=False)
      .agg(
          revenue=("net_sales", "sum"),
          profit=("profit", "sum"),
          orders=("order_id", "nunique")
      )
)
channel_performance["profit_margin_pct"] = (
    channel_performance["profit"] / channel_performance["revenue"] * 100
)

segment_performance = (
    df.groupby("customer_segment", as_index=False)
      .agg(
          revenue=("net_sales", "sum"),
          profit=("profit", "sum"),
          orders=("order_id", "nunique"),
          customers=("customer_id", "nunique")
      )
)
segment_performance["profit_margin_pct"] = (
    segment_performance["profit"] / segment_performance["revenue"] * 100
)

display(channel_performance.sort_values("revenue", ascending=False))
display(segment_performance.sort_values("revenue", ascending=False))

In [ ]:
plt.figure(figsize=(9, 5))
plot_data = segment_performance.sort_values("revenue", ascending=False)
plt.bar(plot_data["customer_segment"], plot_data["revenue"])
plt.title("Revenue by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Revenue")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 9. Customer analytics

We build a customer-level summary to identify high-value customers and understand customer purchasing behavior.

In [ ]:
customer_summary = (
    df.groupby(["customer_id", "customer_name"], as_index=False)
      .agg(
          revenue=("net_sales", "sum"),
          profit=("profit", "sum"),
          orders=("order_id", "nunique"),
          quantity=("quantity", "sum"),
          avg_order_value=("net_sales", "mean"),
          avg_rating=("customer_rating", "mean")
      )
)

customer_summary["profit_margin_pct"] = (
    customer_summary["profit"] / customer_summary["revenue"].replace(0, np.nan) * 100
)

print("Top 10 customers by revenue:")
display(customer_summary.sort_values("revenue", ascending=False).head(10))

print("Top 10 customers by profit:")
display(customer_summary.sort_values("profit", ascending=False).head(10))

In [ ]:
top_customers = customer_summary.sort_values("revenue", ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_customers["customer_name"].astype(str), top_customers["revenue"])
plt.title("Top 10 Customers by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Customer")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Repeat-customer analysis

Repeat customers are useful for understanding retention and customer value.

In [ ]:
repeat_summary = (
    df.groupby("customer_id")
      .agg(
          orders=("order_id", "nunique"),
          revenue=("net_sales", "sum"),
          profit=("profit", "sum")
      )
      .reset_index()
)

repeat_summary["customer_type"] = np.where(
    repeat_summary["orders"] > 1, "Repeat", "One-time"
)

repeat_counts = repeat_summary["customer_type"].value_counts().rename_axis("customer_type").reset_index(name="customers")
repeat_counts["pct"] = repeat_counts["customers"] / repeat_counts["customers"].sum() * 100

display(repeat_counts)

## 11. Product and category analysis

If `order_items` and `product_catalog` were uploaded, we can perform true product-level analysis. This is more reliable than trying to infer a product from the order-level dataset.

In [ ]:
def find_uploaded(prefix):
    return next(
        (f for f in uploaded_names if f.lower().startswith(prefix.lower()) and f.lower().endswith(".csv")),
        None
    )

order_items_file = find_uploaded("order_items")
product_file = find_uploaded("product_catalog")

if order_items_file and product_file:
    order_items = pd.read_csv(order_items_file)
    products = pd.read_csv(product_file)

    oi_products = order_items.merge(
        products,
        on="product_id",
        how="left",
        suffixes=("", "_catalog")
    )

    category_performance = (
        oi_products.groupby("product_category", as_index=False)
        .agg(
            revenue=("net_sales", "sum"),
            profit=("profit", "sum"),
            units=("quantity", "sum"),
            products=("product_id", "nunique")
        )
    )
    category_performance["profit_margin_pct"] = (
        category_performance["profit"] /
        category_performance["revenue"].replace(0, np.nan) * 100
    )

    product_performance = (
        oi_products.groupby(["product_id", "product_name"], as_index=False)
        .agg(
            revenue=("net_sales", "sum"),
            profit=("profit", "sum"),
            units=("quantity", "sum")
        )
    )
    product_performance["profit_margin_pct"] = (
        product_performance["profit"] /
        product_performance["revenue"].replace(0, np.nan) * 100
    )

    print("Category performance:")
    display(category_performance.sort_values("revenue", ascending=False))

    print("Top 10 products by revenue:")
    display(product_performance.sort_values("revenue", ascending=False).head(10))

    print("Top 10 products by profit:")
    display(product_performance.sort_values("profit", ascending=False).head(10))
else:
    print("Supporting product files were not uploaded.")
    print("Upload order_items and product_catalog to run product/category analysis.")

In [ ]:
if "category_performance" in globals():
    top_categories = category_performance.sort_values("revenue", ascending=False)

    plt.figure(figsize=(10, 6))
    plt.barh(top_categories["product_category"], top_categories["revenue"])
    plt.title("Revenue by Product Category")
    plt.xlabel("Revenue")
    plt.ylabel("Category")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
if "product_performance" in globals():
    top_products = product_performance.sort_values("revenue", ascending=False).head(10)

    plt.figure(figsize=(10, 6))
    plt.barh(top_products["product_name"].astype(str), top_products["revenue"])
    plt.title("Top 10 Products by Revenue")
    plt.xlabel("Revenue")
    plt.ylabel("Product")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 12. Marketing performance

Compare marketing channels and campaigns using revenue, profit, and margin. Missing campaign/coupon values are retained as missing rather than being assumed to mean "no campaign".

In [ ]:
marketing = (
    df.groupby("marketing_channel", dropna=False, as_index=False)
      .agg(
          revenue=("net_sales", "sum"),
          profit=("profit", "sum"),
          orders=("order_id", "nunique")
      )
)
marketing["profit_margin_pct"] = (
    marketing["profit"] / marketing["revenue"].replace(0, np.nan) * 100
)

campaign = (
    df.groupby("campaign_name", dropna=False, as_index=False)
      .agg(
          revenue=("net_sales", "sum"),
          profit=("profit", "sum"),
          orders=("order_id", "nunique")
      )
)
campaign["profit_margin_pct"] = (
    campaign["profit"] / campaign["revenue"].replace(0, np.nan) * 100
)

display(marketing.sort_values("revenue", ascending=False))
display(campaign.sort_values("revenue", ascending=False).head(15))

In [ ]:
marketing_plot = marketing.dropna(subset=["marketing_channel"]).sort_values("revenue", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(marketing_plot["marketing_channel"].astype(str), marketing_plot["revenue"])
plt.title("Revenue by Marketing Channel")
plt.xlabel("Marketing Channel")
plt.ylabel("Revenue")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 13. Returns analysis

Returned orders are analyzed by return reason. We do not fill missing return values because missingness can represent orders that were never returned.

In [ ]:
returned = df[df["is_returned"]].copy()

return_reasons = (
    returned.groupby("return_reason", dropna=False)
    .agg(
        returned_orders=("order_id", "nunique"),
        revenue=("net_sales", "sum"),
        profit=("profit", "sum")
    )
    .reset_index()
    .sort_values("returned_orders", ascending=False)
)

display(return_reasons)

plt.figure(figsize=(10, 6))
plt.barh(return_reasons["return_reason"].astype(str), return_reasons["returned_orders"])
plt.title("Returned Orders by Reason")
plt.xlabel("Returned Orders")
plt.ylabel("Return Reason")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 14. Operations and delivery performance

This section looks at delivery status, shipping methods, warehouses, and delivery times.

In [ ]:
delivery_status = (
    df.groupby("delivery_status", dropna=False, as_index=False)
      .agg(
          orders=("order_id", "nunique"),
          revenue=("net_sales", "sum"),
          profit=("profit", "sum")
      )
      .sort_values("orders", ascending=False)
)

shipping = (
    df.groupby("shipping_method", dropna=False, as_index=False)
      .agg(
          orders=("order_id", "nunique"),
          avg_delivery_days=("delivery_days", "mean"),
          avg_estimated_days=("estimated_delivery_days", "mean"),
          revenue=("net_sales", "sum")
      )
)

warehouse = (
    df.groupby("warehouse", dropna=False, as_index=False)
      .agg(
          orders=("order_id", "nunique"),
          avg_delivery_days=("delivery_days", "mean"),
          revenue=("net_sales", "sum"),
          profit=("profit", "sum")
      )
)

display(delivery_status)
display(shipping)
display(warehouse)

## 15. Loss-making orders

Loss-making transactions can reveal discounting problems, high costs, returns, or other profitability issues. We report them rather than deleting them.

In [ ]:
loss_summary = pd.DataFrame({
    "Metric": [
        "Loss-making orders",
        "Total orders",
        "Loss-making order %",
        "Revenue from loss-making orders",
        "Profit from loss-making orders"
    ],
    "Value": [
        loss_orders,
        total_orders,
        loss_order_pct,
        df.loc[df["is_loss_making"], "net_sales"].sum(),
        df.loc[df["is_loss_making"], "profit"].sum()
    ]
})

display(loss_summary)

loss_by_channel = (
    df.groupby("sales_channel", as_index=False)
      .agg(
          loss_orders=("is_loss_making", "sum"),
          total_orders=("order_id", "nunique"),
          loss_profit=("profit", lambda x: x[x < 0].sum())
      )
)

loss_by_channel["loss_order_pct"] = (
    loss_by_channel["loss_orders"] /
    loss_by_channel["total_orders"].replace(0, np.nan) * 100
)

display(loss_by_channel.sort_values("loss_order_pct", ascending=False))

## 16. Best and weakest periods

This creates concise business-ready findings from the calculated results.

In [ ]:
best_revenue_month = monthly.loc[monthly["revenue"].idxmax()]
best_profit_month = monthly.loc[monthly["profit"].idxmax()]
worst_profit_month = monthly.loc[monthly["profit"].idxmin()]

print("Best month by revenue:")
print(best_revenue_month.to_dict())

print("\nBest month by profit:")
print(best_profit_month.to_dict())

print("\nWeakest month by profit:")
print(worst_profit_month.to_dict())

if "category_performance" in globals():
    top_category = category_performance.sort_values("revenue", ascending=False).iloc[0]
    print("\nTop product category by revenue:")
    print(top_category.to_dict())

if "product_performance" in globals():
    top_product = product_performance.sort_values("revenue", ascending=False).iloc[0]
    print("\nTop product by revenue:")
    print(top_product.to_dict())

top_customer = customer_summary.sort_values("revenue", ascending=False).iloc[0]
print("\nTop customer by revenue:")
print(top_customer.to_dict())

top_marketing = marketing.sort_values("revenue", ascending=False).iloc[0]
print("\nTop marketing channel by revenue:")
print(top_marketing.to_dict())

## 17. Portfolio-ready business insights

The following section converts the analysis into concise statements you can use when explaining the project in an interview. The values are generated from the dataset each time the notebook runs.

In [ ]:
print("BUSINESS INSIGHTS")
print("=" * 70)

print(f"1. The dataset contains {total_orders:,} unique orders from {total_customers:,} customers.")
print(f"2. Total net sales are {total_revenue:,.2f}, with total profit of {total_profit:,.2f}.")
print(f"3. Overall profit margin is {profit_margin:.2f}% and average order value is {aov:,.2f}.")
print(f"4. Return rate is {return_rate:.2f}% and cancellation rate is {cancellation_rate:.2f}%.")
print(f"5. {loss_order_pct:.2f}% of orders are loss-making, highlighting an area for profitability investigation.")
print(f"6. The strongest revenue month is {best_revenue_month['order_month']}.")
print(f"7. The strongest profit month is {best_profit_month['order_month']}.")

if "category_performance" in globals():
    print(f"8. The top revenue category is {top_category['product_category']}.")

if "product_performance" in globals():
    print(f"9. The top revenue product is {top_product['product_name']}.")

print(f"10. The highest-revenue customer is {top_customer['customer_name']}.")
print(f"11. The highest-revenue marketing channel is {top_marketing['marketing_channel']}.")

## 18. Export analysis tables for Power BI

The following cell creates clean CSV summary tables. You can download them and use them in Power BI Desktop, or connect Power BI directly to the original CSV files.

Recommended Power BI pages:
1. **Executive Overview** — revenue, profit, margin, AOV, orders, customers, return rate.
2. **Sales & Product Analytics** — monthly trends, category revenue/profit, top products.
3. **Customer Analytics** — customer segments, top customers, repeat vs one-time.
4. **Marketing & Operations** — channels, campaigns, returns, delivery and warehouse performance.

In [ ]:
export_tables = {
    "monthly_performance.csv": monthly,
    "customer_summary.csv": customer_summary,
    "channel_performance.csv": channel_performance,
    "segment_performance.csv": segment_performance,
    "marketing_performance.csv": marketing,
    "campaign_performance.csv": campaign,
    "return_reasons.csv": return_reasons,
    "delivery_status.csv": delivery_status,
    "shipping_performance.csv": shipping,
    "warehouse_performance.csv": warehouse,
}

if "category_performance" in globals():
    export_tables["category_performance.csv"] = category_performance

if "product_performance" in globals():
    export_tables["product_performance.csv"] = product_performance

os.makedirs("/content/ecommerce_analysis_exports", exist_ok=True)

for filename, table in export_tables.items():
    table.to_csv("/content/ecommerce_analysis_exports/" + filename, index=False)

print("Exported files:")
for filename in export_tables:
    print(" -", filename)

## 19. Final project summary

### What this project demonstrates

- Data loading and preparation with Pandas
- Data-quality validation
- KPI calculation
- Time-series sales analysis
- Customer analytics
- Product/category profitability
- Marketing performance analysis
- Return and operations analysis
- Loss-making transaction investigation
- Business insight generation
- Power BI-ready summary tables

### Suggested interview explanation

> "I built an e-commerce analytics project using Python, SQL and Power BI. I analyzed sales, profitability, customers, marketing and operations across more than 100K transactions. I performed data-quality checks, created business KPIs, analyzed monthly trends, identified high-value customers and products, evaluated marketing channels, and investigated returns and loss-making transactions. I then prepared summary datasets for an interactive Power BI dashboard."

**Next step:** Build the Power BI dashboard using the exported summary tables and the original transaction data.